In [ ]:
!pip install scikit-learn-extra

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn_extra.cluster import KMedoids

In [ ]:
# ============================================================
# 1. CARGA DE DATOS SIN LIMPIEZA
# ============================================================
df = pd.read_csv('penguins.csv')
print(f'Dimensiones: {df.shape}')
print(f'\nValores nulos por columna:')
print(df.isnull().sum())
print(f'\nValores unicos en sex: {df["sex"].unique()}')
df.head(10)

In [ ]:
# ============================================================
# 2. PREPARACION MINIMA (solo dropna, sin tratar outliers)
# K-Medoids no puede calcular distancias con NaN,
# pero NO se eliminan outliers: esa es la ventaja del metodo
# ============================================================
df.replace(['NA', '.'], np.nan, inplace=True)
cols_num = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

antes = len(df)
df = df.dropna()
print(f'Filas eliminadas por NaN: {antes - len(df)}')
print(f'Filas restantes: {len(df)}')
print(f'\nSe conservan outliers (flipper=-132, flipper=5000, etc.)')
print(f'Rango flipper_length_mm: [{df["flipper_length_mm"].min()}, {df["flipper_length_mm"].max()}]')
print(f'Rango body_mass_g: [{df["body_mass_g"].min()}, {df["body_mass_g"].max()}]')

In [ ]:
# ============================================================
# 3. ESCALADO
# ============================================================
features = cols_num
x = df[features]
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
print(f'Datos escalados: {x_scaled.shape}')
df.describe()

In [ ]:
# ============================================================
# 4. METODO DEL CODO + SILHOUETTE CON K-MEDOIDS
# ============================================================
K_rango = range(2, 10)
inercia = []
silhouettes = []

for k in K_rango:
    modelo = KMedoids(n_clusters=k, metric='euclidean', random_state=42)
    etiquetas = modelo.fit_predict(x_scaled)
    inercia.append(modelo.inertia_)
    silhouettes.append(silhouette_score(x_scaled, etiquetas))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_rango, inercia, 'ro-', linewidth=2, markersize=8)
ax1.set_title('Metodo del Codo (K-Medoids)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Numero de Clusters (K)')
ax1.set_ylabel('Inercia')
ax1.set_xticks(list(K_rango))
ax1.grid(True, alpha=0.3)
ax1.axvline(x=3, color='blue', linestyle='--', alpha=0.5, label='K=3')
ax1.legend()

ax2.plot(K_rango, silhouettes, 'gs-', linewidth=2, markersize=8)
ax2.set_title('Coeficiente Silhouette (K-Medoids)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Numero de Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_xticks(list(K_rango))
ax2.grid(True, alpha=0.3)
mejor_k = list(K_rango)[np.argmax(silhouettes)]
ax2.axvline(x=mejor_k, color='blue', linestyle='--', alpha=0.5, label=f'Mejor K={mejor_k} ({max(silhouettes):.3f})')
ax2.legend()

plt.tight_layout()
plt.show()

print('Silhouette por K:')
for k, s in zip(K_rango, silhouettes):
    marca = ' <-- optimo' if k == mejor_k else ''
    print(f'  K={k}: {s:.4f}{marca}')

In [ ]:
# ============================================================
# 5. MODELO FINAL: K-MEDOIDS CON K=3
# ============================================================
K_FINAL = 3
kmedoids = KMedoids(n_clusters=K_FINAL, metric='euclidean', random_state=42)
df['cluster'] = kmedoids.fit_predict(x_scaled)

print(f'Modelo: K-Medoids con K={K_FINAL}')
print(f'Silhouette Score: {silhouette_score(x_scaled, df["cluster"]):.4f}')
print(f'Inercia: {kmedoids.inertia_:.2f}')
print(f'\nDistribucion de clusters:')
for c, n in df['cluster'].value_counts().sort_index().items():
    print(f'  Cluster {c}: {n} pinguinos ({n/len(df)*100:.1f}%)')

# Medoids: son puntos REALES del dataset
print(f'\nMedoids (indices en el dataset): {kmedoids.medoid_indices_}')
medoids_orig = scaler.inverse_transform(kmedoids.cluster_centers_)
df_medoids = pd.DataFrame(medoids_orig, columns=cols_num)
df_medoids.index.name = 'cluster'
print(f'\nMedoids (valores reales):')
print(df_medoids.round(2))

In [ ]:
# ============================================================
# 6. PERFIL DE CADA CLUSTER
# ============================================================
print('--- Promedios por Cluster ---')
perfil = df.groupby('cluster')[cols_num].agg(['mean', 'std']).round(2)
print(perfil)

print('\n--- Composicion por Sexo ---')
print(pd.crosstab(df['cluster'], df['sex'], margins=True))

In [ ]:
# ============================================================
# 7. VISUALIZACION: SCATTER PLOTS
# ============================================================
colores = ['#e74c3c', '#2ecc71', '#3498db']
nombres = ['Cluster 0', 'Cluster 1', 'Cluster 2']

pares = [
    ('culmen_length_mm', 'flipper_length_mm', 'Longitud Pico vs Largo Aleta'),
    ('culmen_length_mm', 'culmen_depth_mm', 'Longitud Pico vs Profundidad Pico'),
    ('flipper_length_mm', 'body_mass_g', 'Largo Aleta vs Masa Corporal'),
    ('culmen_length_mm', 'body_mass_g', 'Longitud Pico vs Masa Corporal')
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (xvar, yvar, titulo) in enumerate(pares):
    ax = axes[idx // 2][idx % 2]
    for c in range(K_FINAL):
        grupo = df[df['cluster'] == c]
        ax.scatter(grupo[xvar], grupo[yvar],
                   c=colores[c], label=nombres[c],
                   alpha=0.7, edgecolors='k', linewidth=0.3, s=50)
    # Medoids en escala original
    ix = cols_num.index(xvar)
    iy = cols_num.index(yvar)
    ax.scatter(medoids_orig[:, ix], medoids_orig[:, iy],
               c='black', marker='D', s=200, edgecolors='white', linewidth=2, label='Medoids', zorder=5)
    ax.set_xlabel(xvar)
    ax.set_ylabel(yvar)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('K-Medoids: Agrupamiento SIN limpieza de outliers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8. BOXPLOTS POR CLUSTER
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cols_num):
    datos = [df[df['cluster'] == c][col].values for c in range(K_FINAL)]
    bp = axes[i].boxplot(datos, patch_artist=True, labels=[f'C{c}' for c in range(K_FINAL)])
    for j, box in enumerate(bp['boxes']):
        box.set_facecolor(colores[j])
        box.set_alpha(0.7)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].grid(True, alpha=0.2)

plt.suptitle('Distribucion por Cluster (K-Medoids, datos sin limpiar)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 9. SILHOUETTE POR MUESTRA
# ============================================================
sil_vals = silhouette_samples(x_scaled, df['cluster'])
sil_avg = silhouette_score(x_scaled, df['cluster'])

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for c in range(K_FINAL):
    cluster_sil = np.sort(sil_vals[df['cluster'].values == c])
    y_upper = y_lower + len(cluster_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=colores[c], alpha=0.7, label=f'Cluster {c} (n={len(cluster_sil)})')
    ax.text(-0.05, y_lower + 0.5 * len(cluster_sil), str(c), fontweight='bold', fontsize=12)
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=2, label=f'Promedio: {sil_avg:.3f}')
ax.set_title('Silhouette por Muestra (K-Medoids)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente Silhouette')
ax.set_ylabel('Muestras')
ax.legend(loc='best')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 10. RESUMEN FINAL
# ============================================================
print('='*60)
print('RESUMEN DEL AGRUPAMIENTO (K-MEDOIDS, SIN LIMPIEZA)')
print('='*60)
print(f'Algoritmo: K-Medoids')
print(f'Limpieza de outliers: NO')
print(f'K seleccionado: {K_FINAL}')
print(f'Filas procesadas: {len(df)}')
print(f'Features: {cols_num}')
print(f'Escalado: StandardScaler')
print(f'Inercia: {kmedoids.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(x_scaled, df["cluster"]):.4f}')
print(f'\nVentaja de K-Medoids sobre K-Means:')
print(f'  Los medoids son puntos REALES del dataset, no promedios.')
print(f'  Mas robusto ante outliers como flipper=-132 o flipper=5000.')
print(f'\nInterpretacion de clusters:')
for c in range(K_FINAL):
    g = df[df['cluster'] == c]
    print(f'  Cluster {c}: {len(g)} pinguinos | '
          f'pico={g["culmen_length_mm"].mean():.1f}mm | '
          f'aleta={g["flipper_length_mm"].mean():.1f}mm | '
          f'masa={g["body_mass_g"].mean():.0f}g')